In [1]:
# ==============================================================
# LLAMAINDEX + CHROMADB + LLM + TOOLS
# SINGLE-CELL JUPYTER NOTEBOOK DEMONSTRATION
#
# Application: Intelligent Customer-Support Agent
# ==============================================================

# --------------------------------------------------------------
# 1. Install required packages
# --------------------------------------------------------------

import sys
import subprocess
import os
from getpass import getpass

required_packages = [
    "llama-index",
    "llama-index-llms-openai",
    "llama-index-embeddings-openai",
    "llama-index-vector-stores-chroma",
    "chromadb"
]

print("Installing the required packages...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        *required_packages
    ]
)

print("Packages installed successfully.\n")

Installing the required packages...
Packages installed successfully.



In [2]:
# --------------------------------------------------------------
# 2. Import the required classes
# --------------------------------------------------------------

import chromadb
from datetime import datetime
from typing import Dict, Any

from llama_index.core import (
    Document,
    Settings,
    StorageContext,
    VectorStoreIndex
)

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

from llama_index.core.tools import (
    FunctionTool,
    QueryEngineTool
)

from llama_index.core.agent.workflow import FunctionAgent

In [3]:
from google.colab import userdata
from dotenv import load_dotenv
load_dotenv()

# Get the API key from environment variables
openai_api_key = userdata.get("OPENAI_API_KEY")
import os
os.environ["OPENAI_API_KEY"] = openai_api_key

# ------------------------------------------------------------
# 3. Read the OpenAI API key securely
# ------------------------------------------------------------

#if not os.environ.get("OPENAI_API_KEY"):
#    os.environ["OPENAI_API_KEY"] = getpass(
#        "Enter your OpenAI API key: "
#    )

In [4]:
# --------------------------------------------------------------
# 4. Configure the LLM and embedding model
# --------------------------------------------------------------

Settings.llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0
)

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

Settings.chunk_size = 256
Settings.chunk_overlap = 30


# --------------------------------------------------------------
# 5. Create sample company knowledge-base documents
# --------------------------------------------------------------

knowledge_documents = [

    Document(
        text="""
        Router Troubleshooting Policy

        If the router's red light is blinking, switch off the
        router and disconnect its power supply.

        Wait for thirty seconds, reconnect the power supply and
        restart the router.

        Verify that the fibre-optic cable is firmly connected.

        If the red light continues blinking after two restarts,
        contact technical support.
        """,
        metadata={
            "category": "router support",
            "department": "technical support"
        }
    ),

    Document(
        text="""
        Slow Internet Troubleshooting

        If the internet is slow, first check the Wi-Fi signal
        strength and move closer to the router.

        Disconnect devices that are not currently being used.

        Pause large downloads and software updates.

        Restart both the modem and router.

        A speed test should be performed after restarting the
        equipment.
        """,
        metadata={
            "category": "internet speed",
            "department": "technical support"
        }
    ),

    Document(
        text="""
        Product Warranty Policy

        Routers and modems include a one-year warranty from the
        original date of purchase.

        The customer must provide the original invoice and device
        serial number when requesting replacement.

        Physical damage, water damage, electrical damage caused
        by an external power surge and unauthorized repairs are
        not covered by the warranty.
        """,
        metadata={
            "category": "warranty",
            "department": "customer service"
        }
    ),

    Document(
        text="""
        Password Reset Procedure

        Customers can reset their account password by selecting
        Forgot Password on the login page.

        A password-reset link will be sent to the registered
        email address.

        The link expires after thirty minutes.

        The new password must contain at least eight characters,
        an uppercase letter, a lowercase letter and a number.
        """,
        metadata={
            "category": "password",
            "department": "account support"
        }
    ),

    Document(
        text="""
        Billing Support Policy

        For billing problems, customers must provide their
        customer ID, invoice number and transaction date.

        Billing complaints include duplicate payments, incorrect
        charges, failed transactions and missing receipts.

        The billing team normally responds within two working
        days.
        """,
        metadata={
            "category": "billing",
            "department": "accounts"
        }
    ),

    Document(
        text="""
        Cancellation and Refund Policy

        A subscription can be cancelled from the account portal.

        A full refund is available when cancellation is requested
        within seven days of purchasing a new subscription,
        provided that less than ten percent of the monthly data
        allowance has been used.

        Approved refunds are normally processed within five to
        seven working days.
        """,
        metadata={
            "category": "refund",
            "department": "accounts"
        }
    ),

    Document(
        text="""
        Customer-Support Working Hours

        Technical support is available from Monday to Saturday,
        between 9:00 a.m. and 6:00 p.m.

        Billing support is available from Monday to Friday,
        between 10:00 a.m. and 5:00 p.m.

        Network-outage complaints can be registered through the
        online customer portal at any time.
        """,
        metadata={
            "category": "support hours",
            "department": "customer service"
        }
    )
]

print(
    f"Created {len(knowledge_documents)} "
    "knowledge-base documents."
)



Created 7 knowledge-base documents.


In [5]:

# --------------------------------------------------------------
# 6. Create an in-memory ChromaDB vector database
# --------------------------------------------------------------

chroma_client = chromadb.EphemeralClient()

# Delete an old collection if this notebook cell is run again.
try:
    chroma_client.delete_collection(
        name="customer_support_knowledge"
    )
except Exception:
    pass

chroma_collection = chroma_client.create_collection(
    name="customer_support_knowledge"
)

chroma_vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

storage_context = StorageContext.from_defaults(
    vector_store=chroma_vector_store
)


# --------------------------------------------------------------
# 7. Index the documents in ChromaDB
# --------------------------------------------------------------

print("Creating embeddings and storing them in ChromaDB...")

index = VectorStoreIndex.from_documents(
    knowledge_documents,
    storage_context=storage_context,
    show_progress=True
)

print("ChromaDB vector index created successfully.\n")


# --------------------------------------------------------------
# 8. Create the RAG query engine
# --------------------------------------------------------------

knowledge_query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact"
)


Creating embeddings and storing them in ChromaDB...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/7 [00:00<?, ?it/s]

ChromaDB vector index created successfully.



In [6]:


# --------------------------------------------------------------
# 9. Convert the query engine into an agent tool
# --------------------------------------------------------------

knowledge_base_tool = QueryEngineTool.from_defaults(
    query_engine=knowledge_query_engine,
    name="company_knowledge_base",
    description=(
        "Search the company's support knowledge base for "
        "information about routers, slow internet, passwords, "
        "warranties, refunds, billing and support hours. "
        "Use this tool whenever the user asks about company "
        "policies or troubleshooting procedures."
    )
)


In [7]:


# --------------------------------------------------------------
# 10. Create a simulated order database
# --------------------------------------------------------------

order_database = {
    "ORD1001": {
        "customer": "Arun",
        "product": "Wi-Fi 6 Router",
        "status": "Shipped",
        "expected_delivery": "20 July 2026"
    },

    "ORD1002": {
        "customer": "Meena",
        "product": "Fibre Modem",
        "status": "Processing",
        "expected_delivery": "22 July 2026"
    },

    "ORD1003": {
        "customer": "Karthik",
        "product": "Mesh Wi-Fi System",
        "status": "Delivered",
        "expected_delivery": "Delivered on 14 July 2026"
    }
}


# --------------------------------------------------------------
# 11. Tool 1: Check order status
# --------------------------------------------------------------

def check_order_status(order_id: str) -> str:
    """
    Check the current status of a customer order.

    Args:
        order_id: Order identifier such as ORD1001.

    Returns:
        Order details or an error message.
    """

    normalized_order_id = order_id.strip().upper()

    order = order_database.get(normalized_order_id)

    if order is None:
        return (
            f"No order was found with ID "
            f"{normalized_order_id}. "
            "Please verify the order number."
        )

    return (
        f"Order ID: {normalized_order_id}\n"
        f"Customer: {order['customer']}\n"
        f"Product: {order['product']}\n"
        f"Status: {order['status']}\n"
        f"Expected delivery: "
        f"{order['expected_delivery']}"
    )


order_status_tool = FunctionTool.from_defaults(
    fn=check_order_status,
    name="check_order_status",
    description=(
        "Check an order's status, product and expected delivery "
        "date using an order ID such as ORD1001."
    )
)


# --------------------------------------------------------------
# 12. Tool 2: Calculate price
# --------------------------------------------------------------

def calculate_final_price(
    unit_price: float,
    quantity: int,
    discount_percentage: float = 0,
    tax_percentage: float = 18
) -> Dict[str, Any]:
    """
    Calculate total price after applying discount and tax.

    Args:
        unit_price: Price of one product.
        quantity: Number of products.
        discount_percentage: Discount percentage.
        tax_percentage: Tax percentage; default is 18%.

    Returns:
        A structured price calculation.
    """

    if unit_price < 0:
        raise ValueError("Unit price cannot be negative.")

    if quantity <= 0:
        raise ValueError("Quantity must be greater than zero.")

    if not 0 <= discount_percentage <= 100:
        raise ValueError(
            "Discount percentage must be between 0 and 100."
        )

    if tax_percentage < 0:
        raise ValueError("Tax percentage cannot be negative.")

    subtotal = unit_price * quantity

    discount_amount = (
        subtotal * discount_percentage / 100
    )

    amount_after_discount = (
        subtotal - discount_amount
    )

    tax_amount = (
        amount_after_discount * tax_percentage / 100
    )

    final_price = (
        amount_after_discount + tax_amount
    )

    return {
        "subtotal": round(subtotal, 2),
        "discount_percentage": discount_percentage,
        "discount_amount": round(discount_amount, 2),
        "amount_after_discount": round(
            amount_after_discount,
            2
        ),
        "tax_percentage": tax_percentage,
        "tax_amount": round(tax_amount, 2),
        "final_price": round(final_price, 2)
    }


price_calculator_tool = FunctionTool.from_defaults(
    fn=calculate_final_price,
    name="calculate_final_price",
    description=(
        "Calculate the final purchase price using unit price, "
        "quantity, discount percentage and tax percentage."
    )
)

In [8]:
# --------------------------------------------------------------
# 13. Tool 3: Create a support ticket
# --------------------------------------------------------------

support_tickets = []


def create_support_ticket(
    customer_name: str,
    issue: str,
    priority: str = "medium"
) -> str:
    """
    Create a customer-support ticket.

    Args:
        customer_name: Name of the customer.
        issue: Description of the problem.
        priority: low, medium or high.

    Returns:
        Confirmation containing the generated ticket ID.
    """

    allowed_priorities = {
        "low",
        "medium",
        "high"
    }

    normalized_priority = priority.strip().lower()

    if normalized_priority not in allowed_priorities:
        return (
            "Invalid priority. Use low, medium or high."
        )

    if not customer_name.strip():
        return "Customer name cannot be empty."

    if not issue.strip():
        return "The issue description cannot be empty."

    ticket_number = len(support_tickets) + 1

    ticket_id = f"TKT{ticket_number:04d}"

    ticket = {
        "ticket_id": ticket_id,
        "customer_name": customer_name.strip(),
        "issue": issue.strip(),
        "priority": normalized_priority,
        "created_at": datetime.now().strftime(
            "%d %B %Y, %I:%M %p"
        ),
        "status": "Open"
    }

    support_tickets.append(ticket)

    return (
        "Support ticket created successfully.\n"
        f"Ticket ID: {ticket_id}\n"
        f"Customer: {ticket['customer_name']}\n"
        f"Priority: {normalized_priority.title()}\n"
        f"Status: Open"
    )


ticket_creation_tool = FunctionTool.from_defaults(
    fn=create_support_ticket,
    name="create_support_ticket",
    description=(
        "Create a support ticket when a customer reports a "
        "problem requiring human follow-up. The tool requires "
        "the customer name, issue description and priority."
    )
)


# --------------------------------------------------------------
# 14. Tool 4: List tickets created in this session
# --------------------------------------------------------------

def list_support_tickets() -> str:
    """
    Return all support tickets created during this session.
    """

    if not support_tickets:
        return "No support tickets have been created."

    ticket_lines = []

    for ticket in support_tickets:
        ticket_lines.append(
            f"{ticket['ticket_id']} | "
            f"{ticket['customer_name']} | "
            f"{ticket['priority'].title()} | "
            f"{ticket['status']} | "
            f"{ticket['issue']}"
        )

    return "\n".join(ticket_lines)


list_tickets_tool = FunctionTool.from_defaults(
    fn=list_support_tickets,
    name="list_support_tickets",
    description=(
        "Display all customer-support tickets created during "
        "the current notebook session."
    )
)


# --------------------------------------------------------------
# 15. Create the LlamaIndex tool-using agent
# --------------------------------------------------------------

support_agent = FunctionAgent(
    name="IntelligentCustomerSupportAgent",

    description=(
        "An intelligent customer-support agent that searches "
        "company documentation and uses operational tools."
    ),

    system_prompt="""
You are an intelligent customer-support assistant.

You have access to the following capabilities:

1. Search the company knowledge base stored in ChromaDB.
2. Check customer-order status.
3. Calculate prices, discounts and taxes.
4. Create customer-support tickets.
5. List support tickets created in this session.

Instructions:

- Select the appropriate tool based on the user's request.
- For policy and troubleshooting questions, search the company
  knowledge base before answering.
- Do not invent company policies.
- For order questions, ask for an order ID when one is missing.
- Before creating a support ticket, make sure you have the
  customer's name and a clear issue description.
- Use the price calculator for numerical price calculations.
- Clearly explain the result of the tool.
- When information is not available, say so honestly.
""",

    tools=[
        knowledge_base_tool,
        order_status_tool,
        price_calculator_tool,
        ticket_creation_tool,
        list_tickets_tool
    ],

    llm=Settings.llm
)



In [10]:
# --------------------------------------------------------------
# 16. Display application information
# --------------------------------------------------------------

print("=" * 78)
print("LLAMAINDEX INTELLIGENT CUSTOMER-SUPPORT AGENT")
print("=" * 78)

print(
    """
This agent can:

1. Answer questions from the ChromaDB knowledge base.
2. Troubleshoot router and internet problems.
3. Explain warranty, refund and billing policies.
4. Check simulated order status.
5. Calculate prices, discounts and taxes.
6. Create and list support tickets.

Example questions:

- What should I do if the router red light is blinking?
- What is covered under the product warranty?
- Check the status of order ORD1001.
- Calculate the final price of 3 routers costing ₹5,000 each,
  with a 10% discount and 18% tax.
- Create a high-priority ticket for Ravi because his router
  remains offline after restarting it.
- Show all support tickets.

Type 'exit' to stop.
"""
)




LLAMAINDEX INTELLIGENT CUSTOMER-SUPPORT AGENT

This agent can:

1. Answer questions from the ChromaDB knowledge base.
2. Troubleshoot router and internet problems.
3. Explain warranty, refund and billing policies.
4. Check simulated order status.
5. Calculate prices, discounts and taxes.
6. Create and list support tickets.

Example questions:

- What should I do if the router red light is blinking?
- What is covered under the product warranty?
- Check the status of order ORD1001.
- Calculate the final price of 3 routers costing ₹5,000 each,
  with a 10% discount and 18% tax.
- Create a high-priority ticket for Ravi because his router
  remains offline after restarting it.
- Show all support tickets.

Type 'exit' to stop.



In [11]:
# --------------------------------------------------------------
# 17. Interactive user-agent conversation
# --------------------------------------------------------------

while True:

    user_question = input("You: ").strip()

    if user_question.lower() in {
        "exit",
        "quit",
        "stop"
    }:
        print("\nAgent: Customer-support session ended.")
        break

    if not user_question:
        print(
            "\nAgent: Please enter a valid question.\n"
        )
        continue

    try:
        response = await support_agent.run(
            user_msg=user_question
        )

        print("\nAgent:")
        print(response)
        print("\n" + "-" * 78 + "\n")

    except KeyboardInterrupt:
        print("\nAgent: Session interrupted.")
        break

    except Exception as error:
        print("\nThe request could not be completed.")
        print(
            f"{type(error).__name__}: {error}"
        )
        print()

You: What is covered under the product warranty?

Agent:
The product warranty covers routers and modems for one year from the original date of purchase. To request a replacement, you need to provide the original invoice and the device's serial number. However, the warranty does not cover:

- Physical damage
- Water damage
- Electrical damage from external power surges
- Unauthorized repairs

------------------------------------------------------------------------------

You: exit

Agent: Customer-support session ended.
